### Crosscheck graylog results

In [ ]:
import pymongo
import pandas as pd
from dotenv import load_dotenv
from datetime import datetime, date
import os
import pprint

load_dotenv(override=True)

# mongo_uri = os.getenv("mongo_uri")
mongo_uri_local = os.getenv("mongo_uri_local")

try:
    client = pymongo.MongoClient(mongo_uri)
except NameError:
    print("mongo_uri not available, trying mongo_uri_local")
    client=pymongo.MongoClient(mongo_uri_local)

from pymongo.uri_parser import parse_uri
# print(parse_uri(mongo_uri_local)["options"])
# print(repr(mongo_uri_local))



client.list_database_names()
db = client["klapp-prod"]

### Crosscheck if total sent notification matches with DB entries

In [ ]:
db.list_collection_names()

In [40]:
db["jobs"].find_one({})

{'_id': ObjectId('6a30d80b424b2c6efb3b5053'),
 'type': 'reply',
 'status': 'scheduled',
 'scheduled_at': datetime.datetime(2026, 9, 3, 15, 30),
 'created_by': ObjectId('64e779ffe522340345b3e4c4'),
 'data': {'notification_id': ObjectId('6a2fc6a1758f269490b37806'),
  'reply_id': ObjectId('6a30d7c9809732c4a9e17961'),
  'thread_id': ObjectId('6a30d7c9809732c4a9e17961'),
  'type': 'thread',
  'body': {'from_role': 'parent',
   'read_request': False,
   'recipient_users': [{'thread_started': False,
     'user_id': '68da410095d040fc975cccdf',
     'user_type': 'parent',
     'recipients': []},
    {'user_id': '5f9035fc41edf9004dce3536',
     'user_type': 'teacher',
     'recipients': []},
    {'thread_started': False,
     'user_id': '6499bf5f84c75f01a9b5f287',
     'user_type': 'teacher',
     'recipients': []},
    {'thread_started': True,
     'user_id': '64e779ffe522340345b3e4c4',
     'user_type': 'parent'}],
   'body': '&lt;p>Donnerstag &lt;/p>',
   'files': [],
   'url_links': [],
   '

In [41]:
db["jobs"].distinct("type")

['reply']

In [ ]:
list(db["notification"].aggregate([{"$match": {
                                "draft_id": {"$ne": None}, 
                                "created_at": {"$gte": datetime(2026, 8, 28, 14, 48, 55), 
                                               "$lte": datetime(2026, 8, 28, 14, 49, 10)}}}]))

In [ ]:
total_notificatinos = [{
    "$match": {
        "created_at": {"$gte": datetime(2026, 8, 1), "$lte": datetime(2026, 8, 28)},
        "is_chat": {"$ne": True}
    }},
    {"$group": {
        "_id": 1,
        "total": {"$sum": 1}
    }}
]

In [ ]:
list(db["notification"].aggregate(total_notificatinos))

- Search a specific message to see if 1 sent message gets tracked as multiple notifications for different users.

In [ ]:
find_duplicates = [
    {"$match": {
        "created_at": {"$gte": datetime(2026, 8, 28, 13, 7, 30), "$lte": datetime(2026, 8, 28, 13, 7, 50)},
        "is_chat": {"$ne": True}
    }},
    {"$group": {
        "_id": "$draft_id",  # Feldname geraten – prüfen!
        "count": {"$sum": 1}
    }},
    {"$match": {"count": {"$gt": 1}}}
]

list(db["notification"].aggregate(find_duplicates))

Fanning out is confirmed, lets group the draft id's for the last 30 days and see if they match graylog

In [ ]:
distinct_sends = [
    {"$match": {
        "created_at": {"$gte": datetime(2026, 7, 29), "$lte": datetime(2026, 8, 28)},
        "is_chat": {"$ne": True}
    }},
    {"$group": {
        "_id": "$draft_id"
    }},
    {"$count": "total"}
]

list(db["notification"].aggregate(distinct_sends))